In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 5 - WEEK 11 BAYESIAN OPTIMISATION
# Run from inside week11/
# ============================================================
#
# Strategy:
# - Manually standardise Y.
# - Refit ARD Matern GP including Week 10.
# - Check Week 10 calibration.
# - Keep search concentrated around actual best [1,1,1,1].
# - Explicitly test one-coordinate moves away from the corner.
# - Retain wider/global pools as diagnostics only.
# ============================================================


# ------------------------------------------------------------
# 1. Load cumulative Week 11 data
# ------------------------------------------------------------

X = np.load("function5/initial_inputs.npy")
Y = np.load("function5/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 10 calibration check
# ------------------------------------------------------------
#
# Week 10 selected:
# [1.000000, 1.000000, 0.98997095, 1.000000]
#
# Prior prediction:
# raw mean ≈ 8390.704478
# raw std  ≈ 162.102266
#
# Actual:
# 8472.09307186791
# ------------------------------------------------------------

week10_pred_mean = 8390.704477928135
week10_pred_std = 162.1022663093704
week10_actual = 8472.09307186791

week10_error = (
    week10_actual
    - week10_pred_mean
)

week10_z_error = (
    week10_error
    / week10_pred_std
)

print("\n================================")
print("WEEK 10 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week10_pred_mean)
print("Predicted std :", week10_pred_std)
print("Actual        :", week10_actual)

print("\nPrediction error:")
print(week10_error)

print("\nError / predicted std:")
print(week10_z_error)


# ------------------------------------------------------------
# 3. Manual Y standardisation
# ------------------------------------------------------------

y_mean = np.mean(Y)
y_std = np.std(Y)

Y_scaled = (
    Y - y_mean
) / y_std

best_y_scaled = (
    best_y - y_mean
) / y_std

print("\n================================")
print("Y STANDARDISATION")
print("================================")

print("Y mean:", y_mean)
print("Y std :", y_std)

print("\nBest scaled Y:")
print(best_y_scaled)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(
    X,
    Y_scaled
)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = (
    gp.kernel_.k1.k2.length_scale
)

inverse_ls = (
    1.0 / lengthscales
)

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 5. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. Candidate scales
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


# ------------------------------------------------------------
# 7. Local / wide / global candidates
# ------------------------------------------------------------

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(150000, 4)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(90000, 4)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(70000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)


# ------------------------------------------------------------
# 8. Explicit one-coordinate boundary rays
# ------------------------------------------------------------
#
# Test each coordinate independently away from [1,1,1,1].
#
# This is useful because:
# - x3 slightly below 1 was tested in Week 10
# - x4 slightly below 1 was tested previously
# - we should not assume x1/x2 behave identically
# ------------------------------------------------------------

ray_values = np.linspace(
    0.75,
    1.0,
    20001
)

ray_x1 = np.ones(
    (len(ray_values), 4)
)
ray_x1[:, 0] = ray_values

ray_x2 = np.ones(
    (len(ray_values), 4)
)
ray_x2[:, 1] = ray_values

ray_x3 = np.ones(
    (len(ray_values), 4)
)
ray_x3[:, 2] = ray_values

ray_x4 = np.ones(
    (len(ray_values), 4)
)
ray_x4[:, 3] = ray_values


candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates,
    ray_x1,
    ray_x2,
    ray_x3,
    ray_x4
])


# ------------------------------------------------------------
# 9. Near-duplicate filtering
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.01
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 10. Predictions
# ------------------------------------------------------------

mu_scaled, sigma_scaled = gp.predict(
    candidates,
    return_std=True
)

mu_raw = (
    mu_scaled * y_std
    + y_mean
)

sigma_raw = (
    sigma_scaled * y_std
)


# ------------------------------------------------------------
# 11. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu_scaled,
    sigma_scaled,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print(
    "candidate =",
    candidates[ei_idx]
)

print(
    "raw mean =",
    mu_raw[ei_idx]
)

print(
    "raw std =",
    sigma_raw[ei_idx]
)

print(
    "EI =",
    EI[ei_idx]
)


# ------------------------------------------------------------
# 12. EI sensitivity
# ------------------------------------------------------------

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in [
    0.0,
    0.01,
    0.05,
    0.10
]:

    EI_test = expected_improvement(
        mu_scaled,
        sigma_scaled,
        best_y_scaled,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", xi,
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 13. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(
    mu_scaled
)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print(
    "candidate =",
    candidates[mean_idx]
)

print(
    "raw mean =",
    mu_raw[mean_idx]
)

print(
    "raw std =",
    sigma_raw[mean_idx]
)


# ------------------------------------------------------------
# 14. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n raw mean =", round(mu_raw[idx], 3),
        "\n raw std =", round(sigma_raw[idx], 3),
        "\n scaled UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 15. Distance from actual best
# ------------------------------------------------------------

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    np.linalg.norm(
        candidates[ei_idx]
        - best_x
    )
)

print(
    "Highest mean:",
    np.linalg.norm(
        candidates[mean_idx]
        - best_x
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        np.linalg.norm(
            candidates[idx]
            - best_x
        )
    )


# ------------------------------------------------------------
# 16. Which dimensions remain on upper boundary?
# ------------------------------------------------------------

def upper_boundary_status(
    x,
    tol=0.01
):

    return [
        j + 1
        for j in range(len(x))
        if x[j] >= 1.0 - tol
    ]


print("\n================================")
print("UPPER-BOUNDARY CHECK")
print("================================")

print(
    "EI dimensions near 1:",
    upper_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean dimensions near 1:",
    upper_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.10,
    0.25,
    0.50,
    1.00
]:

    UCB = (
        mu_scaled
        + beta * sigma_scaled
    )

    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta} dimensions near 1:",
        upper_boundary_status(
            candidates[idx]
        )
    )


# ------------------------------------------------------------
# 17. Best point on each explicit ray
# ------------------------------------------------------------

print("\n================================")
print("BEST PREDICTED MEAN ON EACH RAY")
print("================================")

ray_names = [
    "x1 ray",
    "x2 ray",
    "x3 ray",
    "x4 ray"
]

ray_arrays = [
    ray_x1,
    ray_x2,
    ray_x3,
    ray_x4
]

for name, ray in zip(
    ray_names,
    ray_arrays
):

    ray_distance, _ = tree.query(
        ray,
        k=1
    )

    ray_valid = ray[
        ray_distance > 0.01
    ]

    if len(ray_valid) == 0:
        print(
            name,
            ": no admissible points"
        )
        continue

    ray_mu_scaled, ray_sigma_scaled = gp.predict(
        ray_valid,
        return_std=True
    )

    ray_mu_raw = (
        ray_mu_scaled * y_std
        + y_mean
    )

    ray_sigma_raw = (
        ray_sigma_scaled * y_std
    )

    idx = np.argmax(
        ray_mu_scaled
    )

    print(
        name,
        "\n candidate =",
        ray_valid[idx],
        "\n raw mean =",
        ray_mu_raw[idx],
        "\n raw std =",
        ray_sigma_raw[idx],
        "\n distance from best =",
        np.linalg.norm(
            ray_valid[idx]
            - best_x
        ),
        "\n"
    )

DATA
X shape: (30, 4)
Y shape: (30,)

Current best:
[1. 1. 1. 1.] -> 8662.4825

Y range:
min = 0.1129397953712203
max = 8662.4825
std = 3125.3922406186825

WEEK 10 CALIBRATION CHECK
Predicted mean: 8390.704477928135
Predicted std : 162.1022663093704
Actual        : 8472.09307186791

Prediction error:
81.38859393977509

Error / predicted std:
0.5020817770952434

Y STANDARDISATION
Y mean: 2118.9171406292353
Y std : 3125.3922406186825

Best scaled Y:
2.093678122805937


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 25 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(



GP FIT

Fitted kernel:
1.65**2 * Matern(length_scale=[1.2, 1.25, 1.22, 2], nu=2.5) + WhiteKernel(noise_level=0.00135)

ARD lengthscales:
[1.2027214  1.24619103 1.21832839 2.        ]

Normalised inverse-lengthscale sensitivity:
[0.28139935 0.27158357 0.27779457 0.16922251]

CANDIDATE SCALES
Local widths:
[0.08 0.08 0.08 0.08]

Wide widths:
[0.15 0.15 0.15 0.15]

Candidates after duplicate filtering:
354906

PRIMARY EI
candidate = [1.         1.         1.         0.36751931]
raw mean = 5903.792564551805
raw std = 1113.1855716497587
EI = 0.0007635631171185398

EI SENSITIVITY

xi = 0.0 
 candidate = [1.         1.         1.         0.36751931] 
 raw mean = 5903.793 
 raw std = 1113.186 
 EI = 0.00076356 

xi = 0.01 
 candidate = [1.         1.         1.         0.36751931] 
 raw mean = 5903.793 
 raw std = 1113.186 
 EI = 0.00070008 

xi = 0.05 
 candidate = [1.         1.         1.         0.36751931] 
 raw mean = 5903.793 
 raw std = 1113.186 
 EI = 0.00049138 

xi = 0.1 
 candidat

In [2]:
# ============================================================
# FUNCTION 5 - PORTAL-SAFE x2 RAY CHECK
# ============================================================

# Six-decimal points just beyond 0.01 from [1,1,1,1]
x2_values = np.arange(
    0.989999,
    0.970000,
    -0.000001
)

safe_ray = np.ones(
    (len(x2_values), 4)
)

safe_ray[:, 1] = x2_values


# Check distance using the ACTUAL six-decimal submitted points
safe_ray = np.round(
    safe_ray,
    6
)

distance, _ = tree.query(
    safe_ray,
    k=1
)

safe_ray = safe_ray[
    distance > 0.01
]


# GP predictions
safe_mu_scaled, safe_sigma_scaled = gp.predict(
    safe_ray,
    return_std=True
)

safe_mu_raw = (
    safe_mu_scaled * y_std
    + y_mean
)

safe_sigma_raw = (
    safe_sigma_scaled * y_std
)


# Highest posterior mean
safe_idx = np.argmax(
    safe_mu_scaled
)

safe_candidate = safe_ray[
    safe_idx
]

print("================================")
print("PORTAL-SAFE x2 RAY")
print("================================")

print("candidate =", safe_candidate)

print(
    "raw mean =",
    safe_mu_raw[safe_idx]
)

print(
    "raw std =",
    safe_sigma_raw[safe_idx]
)

print(
    "distance from current best =",
    np.linalg.norm(
        safe_candidate - best_x
    )
)

print(
    "\nPortal:",
    "-".join(
        f"{x:.6f}"
        for x in safe_candidate
    )
)

PORTAL-SAFE x2 RAY
candidate = [1.       0.989999 1.       1.      ]
raw mean = 8407.005792278625
raw std = 132.82485810592786
distance from current best = 0.010001000000000038

Portal: 1.000000-0.989999-1.000000-1.000000
